In [2]:
import pandas as pd

In [3]:
df = pd.read_excel('sample_data.xlsx')

print(df)

  Customer Number                                 Client name Service date 1  \
0  PCLCUSTOMER995                        Goldman sachs_Domlur     2025-09-27   
1             123  24-7 Intouch India Pvt. Ltd_ElectronicCity     2026-01-13   

  Service date 2  Service date 3  Service date 4 Service date 5  \
0      2025-10-25      2025-11-29     2025-12-27     2026-01-31   
1      2026-01-27      2026-02-11     2026-02-23            NaT   

  Service date 6 Service type  
0     2026-02-28      Monthly  
1            NaT  Fortnightly  


In [5]:
SERVICE_GAP = {
    "Weekly": 7,
    "Fortnightly": 14,
    "Monthly": 30,      # approximate
    "10 Days": 10
}

In [6]:
def calculate_on_time_percentage(row):
    service_cols = [col for col in row.index if "Service date" in col]
    dates = pd.to_datetime(row[service_cols]).dropna().sort_values().tolist()
    
    service_type = row["Service type"]
    expected_gap = SERVICE_GAP.get(service_type, None)
    
    if not dates or expected_gap is None:
        return None
    
    on_time_count = 1  # first service always on-time
    total_services = len(dates)
    
    for i in range(1, len(dates)):
        actual_gap = (dates[i] - dates[i-1]).days
        
        # tolerance ±3 days
        if abs(actual_gap - expected_gap) <= 3:
            on_time_count += 1
    
    return (on_time_count / total_services) * 100

In [7]:
df["on_time_percentage"] = df.apply(calculate_on_time_percentage, axis=1)

# Service On Time (%) -> Required in report

In [8]:
df

,Customer Number,Client name,Service date 1,Service date 2,Service date 3,Service date 4,Service date 5,Service date 6,Service type,on_time_percentage
0,PCLCUSTOMER995,Goldman sachs_Domlur,2025-09-27,2025-10-25,2025-11-29,2025-12-27,2026-01-31,2026-02-28,Monthly,66.666667
1,123,24-7 Intouch India Pvt. Ltd_ElectronicCity,2026-01-13,2026-01-27,2026-02-11,2026-02-23,NaT,NaT,Fortnightly,100.000000


In [99]:
df_complaints = pd.read_excel('sample_complain_data.xlsx')
df_complaints

,Customer Number,Client name,City Name,Complain Type,Complain Date,Complain Resolve Date
0,NaN,: L’Oréal India Pvt. Ltd,Mumbai,PCL-C-VC,2026-03-19 11:03:00,2026-03-23 11:03:00


In [100]:
def calculate_resolution_time(df):
    df["Complain Date"] = pd.to_datetime(df["Complain Date"])
    df["Complain Resolve Date"] = pd.to_datetime(df["Complain Resolve Date"])
    
    # Resolution time in hours
    df["resolution_time_hours"] = (
        df["Complain Resolve Date"] - df["Complain Date"]
    ).dt.total_seconds() / 3600
    
    # Resolution time in days
    df["resolution_time_days"] = (
        df["Complain Resolve Date"] - df["Complain Date"]
    ).dt.days
    
    return df

In [101]:
df_complaints.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Customer Number        0 non-null      float64       
 1   Client name            1 non-null      str           
 2   City Name              1 non-null      str           
 3   Complain Type          1 non-null      str           
 4   Complain Date          1 non-null      datetime64[us]
 5   Complain Resolve Date  1 non-null      datetime64[us]
dtypes: datetime64[us](2), float64(1), str(3)
memory usage: 180.0 bytes


In [102]:
df_complaints["Complain Date"] = pd.to_datetime(df_complaints["Complain Date"])
df_complaints["Complain Resolve Date"] = pd.to_datetime(df_complaints["Complain Resolve Date"])

df_complaints["Resolution TAT (hours)"] = (
    df_complaints["Complain Resolve Date"] - df_complaints["Complain Date"]
).dt.total_seconds() / 3600

In [103]:
df_complaints["Resolution TAT (days)"] = (
    df_complaints["Complain Resolve Date"] - df_complaints["Complain Date"]
).dt.days

# Complaints Resolution TAT -> Required in report

In [104]:
df_complaints

,Customer Number,Client name,City Name,Complain Type,Complain Date,Complain Resolve Date,Resolution TAT (hours),Resolution TAT (days)
0,NaN,: L’Oréal India Pvt. Ltd,Mumbai,PCL-C-VC,2026-03-19 11:03:00,2026-03-23 11:03:00,96.0,4


In [105]:
outstanding = pd.read_excel("Aging as of 2303.xlsx")

outstanding.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,80825871.79,75131212.43,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,Customer Number,Customer Name,Invoice#,Subclient,Order Number,SO Number,Status,Payment Terms,Invoice Date,Due Date,...,Total,Balance,Account Person Email,Customer Email,Mobile Phone,Subject,Place of Supply,Shipping City,Shipping Street 1,Shipping Street 2
1,PCLCUSTOMER995,Dnyansagar Institute of Management and Researc...,PCL-2021441,NaN,NaN,NaN,Overdue,Due on Receipt,02/04/2023,02/04/2023,...,1239,1239,skppurchase.marathe1962@gmail.com,NaN,NaN,Padcare MHM service bill - May -2022,Maharashtra,Pune,"SKP Campus, Baner - Balewadi Rd, Laxman Nagar,...",NaN
2,PCLCUSTOMER475,Mumbai B.Ed. College for Women,PCL-2021848,NaN,NaN,NaN,Overdue,Due on Receipt,02/04/2023,02/04/2023,...,2000,2000,NaN,m.bedcollege@gmail.com,NaN,MHM-Service August-2022,Maharashtra,Pali,NaN,NaN
3,PCLCUSTOMER188,JSW Steel Limited,PCL-2021846,NaN,NaN,NaN,Overdue,Due on Receipt,02/04/2023,02/04/2023,...,2478,2478,NaN,milind.rode@jsw.in,NaN,Padcare Vending Machine Rent,Maharashtra,Mumbai,"JSW Center, Behind MMRD Ground, BKC, Bandra (E),",NaN
4,PCLCUSTOMER475,Mumbai B.Ed. College for Women,PCL-20211016,NaN,NaN,NaN,Overdue,Due on Receipt,02/04/2023,02/04/2023,...,2000,2000,NaN,m.bedcollege@gmail.com,NaN,MHM-Service September -2022,Maharashtra,Pali,NaN,NaN


In [107]:
outstanding.columns = outstanding.iloc[0]   # set first row as header
outstanding = outstanding[1:].reset_index(drop=True)

In [108]:
df_new = outstanding[
    [
        "Customer Number",
        "Customer Name",
        "Status",
        "Payment Terms",
        "Invoice Date",
        "Due Date",
        "Total",
        "Balance",
    ]
].copy()

In [109]:
df_new["Invoice Date"] = pd.to_datetime(df_new["Invoice Date"], errors="coerce")
df_new["Due Date"] = pd.to_datetime(df_new["Due Date"], errors="coerce")

df_new["Total"] = pd.to_numeric(df_new["Total"], errors="coerce")
df_new["Balance"] = pd.to_numeric(df_new["Balance"], errors="coerce")

# Outstanding customers -> Required in report

In [110]:
df_new

,Customer Number,Customer Name,Status,Payment Terms,Invoice Date,Due Date,Total,Balance
0,PCLCUSTOMER995,Dnyansagar Institute of Management and Researc...,Overdue,Due on Receipt,2023-02-04,2023-02-04,1239.00,1239.00
1,PCLCUSTOMER475,Mumbai B.Ed. College for Women,Overdue,Due on Receipt,2023-02-04,2023-02-04,2000.00,2000.00
2,PCLCUSTOMER188,JSW Steel Limited,Overdue,Due on Receipt,2023-02-04,2023-02-04,2478.00,2478.00
3,PCLCUSTOMER475,Mumbai B.Ed. College for Women,Overdue,Due on Receipt,2023-02-04,2023-02-04,2000.00,2000.00
4,PCLCUSTOMER996,Shri Martand Bhairav Adhyapak Mahavidyalaya,Overdue,Due on Receipt,2023-02-04,2023-02-04,525.00,525.00
...,...,...,...,...,...,...,...,...
9887,PCLCUSTOMER2845,T-Hub Foundation,Sent,Net 45,NaT,2026-07-05,15000.00,15000.00
9888,PCLCUSTOMER2944,GHX India Pvt Ltd,Sent,Net 30,NaT,NaT,3835.00,3835.00
9889,PCLCUSTOMER1929,TE CONNECTIVITY INDIA PVT.LTD.,Sent,Net 30,NaT,NaT,26489.82,26489.82
9890,PCLCUSTOMER412,Hero MotoCorp Ltd - Tirupati,Sent,Net 30,NaT,NaT,41536.00,41536.00


In [81]:
impact_data = pd.read_excel("Impact report_Template.xlsx")

In [82]:
impact_data.head()

,Unique,Unique.1,Month,Zone,City,Client name,Branch,Unique code at branch level,Pads collected,Total Pads collected,Total carbon emission conserved Kg,Total landfill area saved Ltr,Service type,Client User id,Client password,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,46048Blue Dart Express LimitedBlue Dart Expres...,False,2026-01-26,NaN,Indore,Blue Dart Express Limited,Blue Dart Express Limited_VijayNagarIndore,NaN,890,35600,1904.6,17800,Monthly,NaN,NaN,NaN,NaN,NaN,NaN
1,46048Blue Dart Express LimitedBlue Dart Expres...,False,2026-01-26,NaN,Indore,Blue Dart Express Limited,Blue Dart Express Limited_VijayNagarIndore,NaN,890,35600,1904.6,17800,Monthly,NaN,NaN,NaN,NaN,NaN,NaN
2,46048Dxc Technology India Pvt LtdDxc Technolog...,False,2026-01-26,NaN,Indore,Dxc Technology India Pvt Ltd,Dxc Technology India Pvt Ltd_BrillantTitanium,NaN,1900,76000,4066,38000,Monthly,NaN,NaN,NaN,NaN,NaN,NaN
3,46048Globant India Private LimitedGlobant Indi...,False,2026-01-26,NaN,Indore,Globant India Private Limited,Globant India Private Limited_BrillantSapphire,NaN,3000,120000,6420,60000,Monthly,NaN,NaN,NaN,NaN,NaN,NaN
4,46048Indian Institute Of ManagementIndian Inst...,False,2026-01-26,NaN,Indore,Indian Institute Of Management,Indian Institute Of Management_Indore,NaN,14600,584000,31244,292000,Monthly,NaN,NaN,NaN,NaN,NaN,NaN


In [83]:
def prepare_report(df):
    # Select required columns
    df = df[
        [
            "Client name",
            "Branch",
            "Month",
            "Pads collected",
            "Total Pads collected",
        ]
    ].copy()
    
    # Ensure types
    df["Month"] = pd.to_datetime(df["Month"])
    df["Pads collected"] = pd.to_numeric(df["Pads collected"], errors="coerce")
    df["Total Pads collected"] = pd.to_numeric(df["Total Pads collected"], errors="coerce")
    
    # Drop duplicates (your sample shows duplicates)
    df = df.drop_duplicates()
    
    # --- Calculations ---
    
    # Monthly metrics
    df["carbon_emission_conserved_kg"] = df["Pads collected"] * 2.14
    df["landfill_saved_ltr_month"] = df["Pads collected"] * 0.5
    
    # Total landfill saved (based on cumulative pads)
    df["total_landfill_saved_ltr"] = df["Total Pads collected"] * 0.5
    
    return df

In [84]:
df_report = prepare_report(impact_data)

# Impact Data -> Required in report

In [85]:
df_report

,Client name,Branch,Month,Pads collected,Total Pads collected,carbon_emission_conserved_kg,landfill_saved_ltr_month,total_landfill_saved_ltr
0,Blue Dart Express Limited,Blue Dart Express Limited_VijayNagarIndore,2026-01-26,890.0,35600.0,1904.6,445.0,17800.0
2,Dxc Technology India Pvt Ltd,Dxc Technology India Pvt Ltd_BrillantTitanium,2026-01-26,1900.0,76000.0,4066.0,950.0,38000.0
3,Globant India Private Limited,Globant India Private Limited_BrillantSapphire,2026-01-26,3000.0,120000.0,6420.0,1500.0,60000.0
4,Indian Institute Of Management,Indian Institute Of Management_Indore,2026-01-26,14600.0,584000.0,31244.0,7300.0,292000.0
5,LTIMindtree,LTIMindtree_VijayNagar,2026-01-26,2770.0,110800.0,5927.8,1385.0,55400.0
...,...,...,...,...,...,...,...,...
6139,NaN,Visa Consolidated Support\nServices (India) Pr...,2026-02-26,9400.0,376000.0,20116.0,4700.0,188000.0
6140,NaN,Visa Consolidated Support\nServices (India) Pr...,2026-02-26,9440.0,377600.0,20201.6,4720.0,188800.0
6141,NaN,WNS Global Services Private Limited_DLFITSector30,2026-03-26,8100.0,324000.0,17334.0,4050.0,162000.0
6142,NaN,WNS Global Services Private Limited_WorldtechPark,2026-03-26,6100.0,244000.0,13054.0,3050.0,122000.0
